# 01 — Data Cleaning

## Purpose
Load the raw-but-concatenated data (`datanomodifier.csv`) and apply all **cleaning and
standardization** steps before any analysis begins.

After this notebook you have a clean, consistent dataframe called `df` that is ready
to be analyzed in notebooks 02–05.

## Input
`data/intermediate/datanomodifier.csv` — produced by `00_data_pipeline.ipynb`

## Output
No file is saved here. The cleaned `df` is the result. The analysis notebooks
each reload and re-apply these same cleaning steps.

## Run order
Run after `00_data_pipeline.ipynb`.

In [ ]:
import os

# ─── PATH CONFIGURATION ───────────────────────────────────────────────
# Option A — After cloning the repo (default, USE_GITHUB = False)
#   git clone https://github.com/DiegoLarrieta/PanemReto
#   cd PanemReto/notebooks
#   jupyter notebook
#   Paths resolve automatically — no changes needed.
#
# Option B — Read directly from GitHub, e.g. Google Colab (USE_GITHUB = True)
#   Works for notebooks 07 (processed branch CSVs are on GitHub).
#   Notebooks 00-06 need the intermediate files which are NOT in the repo
#   (too large). Run 00_data_pipeline.ipynb locally first to generate them.
# ──────────────────────────────────────────────────────────────────────────

USE_GITHUB   = False
GITHUB_BASE  = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — generate locally with 00_data_pipeline.ipynb
else:
    PROCESSED_DIR    = PROCESSED_DIR
    WEATHER_DIR      = WEATHER_DIR
    RAW_DIR          = RAW_DIR
    INTERMEDIATE_DIR = INTERMEDIATE_DIR


## Step 1 — Imports and paths

## Step 2 — Load the data

`datanomodifier.csv` is the output of `00_data_pipeline.ipynb`. It already has:
- Modifier rows removed
- `cold_or_warm` column added
- `day_part` column added
- Weather (`tavg`) merged in

We use `low_memory=False` because some columns have mixed types across files.

In [ ]:
input_path = os.path.join(INTERMEDIATE_DIR, "datanomodifier.csv")
df = pd.read_csv(input_path, low_memory=False)

print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

## Step 3 — Convert datetime columns

Even though `00_data_pipeline` already parsed these, they get stored as strings in CSV.
We re-parse them here so every notebook has proper `datetime64` types.

- `operating_date` = the business date of the sale (what day the branch was operating)
- `closing_time` = when the ticket was closed at the POS terminal
- `captured_time` = when the individual item was added to the order (most granular timestamp)

In [ ]:
datetime_cols = ["operating_date", "closing_time", "captured_time"]
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Datetime columns after conversion:")
print(df[datetime_cols].dtypes)

## Step 4 — Fix the `is_modifier` column

In the raw POS exports this column contains a mix of: `True`, `False`, `'True'`, `'False'`,
and `NaN`. We standardize it to a proper Python boolean.

Note: at this stage `is_modifier=True` rows should already be removed, but this safeguard
ensures any that slipped through are handled correctly.

In [ ]:
df["is_modifier"] = (
    df["is_modifier"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
    .fillna(False)
    .astype(bool)
)

print("is_modifier value counts:")
print(df["is_modifier"].value_counts())

## Step 5 — Remove beverage groups

**Why remove beverages?**
The two beverage groups (`CAFE Y BEBIDAS CALIENTES` and `JUGOS Y BEBIDAS FRIAS`) dominate
sales volume and would distort the product rankings. More importantly, the business goal
is to forecast **food product demand** for daily stock planning — beverages are either
prepared on-demand or have different supply chain logic.

We completely remove these groups from the dataframe before any analysis.

In [ ]:
beverage_groups = ["CAFE Y BEBIDAS CALIENTES", "JUGOS Y BEBIDAS FRIAS"]

rows_before = len(df)
df = df[~df["group"].isin(beverage_groups)].copy()
rows_removed = rows_before - len(df)

print(f"Rows before removing beverages: {rows_before:,}")
print(f"Rows removed (beverages):       {rows_removed:,}")
print(f"Rows remaining:                 {len(df):,}")
print()
print("Remaining product groups:")
print(df["group"].value_counts())

## Step 6 — Standardize branch names

The POS system was reconfigured at some point and some branches appear under two different
names in the data (e.g., 'Panem - Hotel Kavia N' and 'Panem - Hotel Kavia' are the same branch).

We map all legacy names to the canonical current names so every analysis groups them correctly.

In [ ]:
sucursal_map = {
    "Panem - Hotel Kavia N"      : "Panem - Hotel Kavia",
    "Panem - Plaza QIN N"        : "Panem - Plaza QIN",
    "Panem - Hospital Zambrano N": "Panem - Hospital Zambrano",
    "Panem - La Carreta N"       : "Panem - Carreta",
}

df["sucursal"] = df["sucursal"].replace(sucursal_map)

print(f"Unique branches: {df['sucursal'].nunique()}")
print(sorted(df["sucursal"].dropna().unique()))

## Step 7 — Fix product name typos

A small number of product names have spelling errors that cause the same product
to appear as two separate items in groupby operations.

We correct known typos here. More may be discovered during EDA and added to this list.

In [ ]:
name_fixes = {
    "SANDWITCH ENSALADA POLLO": "SANDWICH ENSALADA POLLO",
}

df["item"] = df["item"].replace(name_fixes)
print(f"Applied {len(name_fixes)} name fix(es)")

## Step 8 — Data quality check

A quick sanity check before we proceed to analysis: shape, missing values,
and a sample of rows.

In [ ]:
print(f"Final shape: {df.shape}")
print()
print("Missing values per column:")
missing = df.isna().sum()
print(missing[missing > 0])

In [ ]:
df.head(10)

## Summary

After this notebook, `df` contains:
- ✅ Proper datetime types
- ✅ `is_modifier` as boolean
- ✅ Beverages removed
- ✅ Branch names standardized (7 unique branches)
- ✅ Product name typos fixed

**Next step:** `02_eda_sales.ipynb` — explore what sells, where, and how much.